# Data Structures (with `pandas`)

This notebook will teach you the basics of `pandas`, Python's go-to library for
working with tables (a.k.a. DataFrames).

In [ ]:
import pandas as pd

URL = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2023/2023-10-17/'

## Declaring DataFrames

A DataFrame is just flat data (rows and columns) plus an *index* that labels
each row. Let's build the exact same table in three different ways.

### From a Dictionary

Each key is a column name, and each value is a list with that column's values.

In [ ]:
# Each key-value pair is a column
df_dict = pd.DataFrame({'col1': [1, 2, 3], 'col2': [4, 5, 6]})
df_dict

### From a List of Rows

Each element of the list is a row. Rows don't know the names of their columns,
so we have to pass them ourselves using `columns`.

In [ ]:
# Each tuple is a row (lists work too!)
df_rows = pd.DataFrame(data=[(1, 4), (2, 5), (3, 6)], columns=['col1', 'col2'])
df_rows

### From a List of Dictionaries

Each dictionary is a *record* (a row), and its keys tell pandas which column
each value belongs to. This is how a lot of data looks when it comes from the
web (e.g., JSON from an API).

In [ ]:
records = [
    {'col1': 1, 'col2': 4},
    {'col1': 2, 'col2': 5},
    {'col1': 3, 'col2': 6},
]
df_records = pd.DataFrame(records)
df_records

In [ ]:
# Check that all three are truly the same table
print(df_dict.equals(df_rows))
print(df_dict.equals(df_records))

Notice the `0, 1, 2` on the left of every table? That's the **index**. pandas
adds one automatically if we don't give it one. The values are just flat data;
the index is what lets us find rows.

## Indexes and Slicing

Time for some real data! We'll use the Taylor Swift dataset from
[TidyTuesday](https://github.com/rfordatascience/tidytuesday/tree/main/data/2023/2023-10-17).
We can read it directly from a URL, so there's nothing to download.

In [ ]:
# Read the albums table directly from the web
albums = pd.read_csv(url + 'taylor_albums.csv')
display(albums.head())

In [ ]:
# The default index (0, 1, ..., 13) is not very informative. Let's use album
# names instead
albums_idx = albums.set_index('album_name')
display(albums_idx.head())

### `.loc`

Use `.loc` to select by **label** (i.e., index values and column names).

In [ ]:
# A single row
print(albums_idx.loc['folklore'])

# A single cell: [row, column]
print(albums_idx.loc['folklore', 'metacritic_score'])

In [ ]:
# A slice of rows and a list of columns
albums_idx.loc['1989':'folklore', ['album_release', 'metacritic_score']]

### `.iloc`

Use `.iloc` to select by **position** (0, 1, 2, ...), just like we did with
lists in the previous notebook.

In [ ]:
# First row
print(albums_idx.iloc[0])

# Rows in positions 6, 7 and 8, and columns in positions 1 and 2
albums_idx.iloc[6:9, [1, 2]]

**Watch out!** `.loc` slices *include* the last label, but `.iloc` slices
*exclude* the last position.

In [ ]:
# 1989 is in position 6 and folklore is in position 9
print(albums_idx.loc['1989':'folklore'].shape)  # 4 rows
print(albums_idx.iloc[6:9].shape)  # 3 rows! Position 9 is excluded

Our index is now made of album names, so there is no row labeled `0` anymore.
Let's see what happens if we try to grab it with `.loc`.

In [ ]:
albums_idx.loc[0]

Try to select the `user_score` of every album from `Lover` to `Midnights` using
`.loc`. Then, do the same using `.iloc`.

In [ ]:
# Your code here

In [ ]:
print(albums_idx.loc['Lover':'Midnights', 'user_score'])
print(albums_idx.iloc[8:14, 3])

### Boolean Masks

A **mask** is a Series of `True`/`False` values (one per row). We can use a mask
to filter rows based on conditions. When combined with `.loc`, we can slice both
the matching rows and specific columns at the same time!

In [ ]:
# Create a boolean mask: which albums have a metacritic score of 80 or more?
mask = albums_idx['metacritic_score'] >= 80
print(mask)

# Slice rows where mask is True, along with specific columns using .loc
albums_idx.loc[mask, ['user_score', 'metacritic_score']]

## Creating and Updating Columns

To declare a new column, assign values to a column name that doesn't exist yet.

In [ ]:
# A constant: every row gets the same value
albums_idx['artist'] = 'Taylor Swift'

# From other columns: user scores are 0-10, so put them on a 0-100 scale
albums_idx['user_score_100'] = albums_idx['user_score'] * 10
albums_idx

To update a column, assign values to a column name that already exists. We can
also update only the rows that meet a condition (remember `if` statements?) by
passing the condition to `.loc`.

In [ ]:
# Overwrite `album_release` (text) with actual dates
albums_idx['album_release'] = pd.to_datetime(albums_idx['album_release'])

# Init a column with a default value, then update the rows that meet a condition
albums_idx['acclaimed'] = False
albums_idx.loc[albums_idx['metacritic_score'] >= 80, 'acclaimed'] = True
albums_idx[['album_release', 'metacritic_score', 'acclaimed']]

Try to declare a column named `score_gap` that contains the difference between
`metacritic_score` and `user_score_100`.

In [ ]:
# Your code here

In [ ]:
albums_idx['score_gap'] = albums_idx['metacritic_score'] - albums_idx['user_score_100']
albums_idx[['metacritic_score', 'user_score_100', 'score_gap']]

## Joins

Joins combine two tables using a column they have in common (the *key*). Let's
load the songs table, which shares `album_name` with `albums`.

In [ ]:
songs = pd.read_csv(URL + 'taylor_album_songs.csv')
songs.head()

In [ ]:
# Keep things small: 7 albums (Fearless to folklore) and only their release date
albums_small = albums.iloc[3:10, [0, 2]]

# Only the first track of every album
first_tracks = songs.loc[songs['track_number'] == 1, ['album_name', 'track_name']]

print(albums_small)
print(first_tracks)

### Left Join

Keeps **every row of the left table** and brings in matching values from the
right table. If there is no match, we get `NaN`.

Why are `Fearless` and `Red` empty? The songs table only includes
*Taylor's Version* of re-released albums. We stan artists owning their songs!

In [ ]:
pd.merge(albums_small, first_tracks, on='album_name', how='left')

### Inner Join

Keeps **only the rows that match** in both tables.

In [ ]:
pd.merge(albums_small, first_tracks, on='album_name', how='inner')

### Outer Join

Keeps **every row of both tables**, matched or not.

In [ ]:
pd.merge(albums_small, first_tracks, on='album_name', how='outer')

To sum up:

| Join | Keeps | Rows here |
| :--- | :--- | :---: |
| `left` | All rows of the left table | 7 |
| `inner` | Only rows that match in both tables | 5 |
| `outer` | All rows of both tables | 12 |